# 🌾 Crop Yield Prediction — ML Pipeline
### Architecture: Random Forest · Gradient Boosting · Ridge Regression
**Target:** `Yield_kg_per_ha` &nbsp;|&nbsp; **Dataset:** 50,765 records · 1966–2017 · 4 crops · India
---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, json, os
warnings.filterwarnings('ignore')
%matplotlib inline
plt.rcParams['figure.dpi'] = 120

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import joblib

os.makedirs('model_output', exist_ok=True)
print("✅ Libraries loaded")

## 1. Load & Explore Data

In [ ]:
df = pd.read_csv('Custom_Crops_yield_Historical_Dataset.csv')
print(f"Shape: {df.shape}")
print(f"Crops: {sorted(df['Crop'].unique())}")
print(f"Years: {df['Year'].min()} – {df['Year'].max()}")
print(f"Missing values: {df.isnull().sum().sum()}")
df.head()

In [ ]:
df.describe().round(2)

## 2. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for crop, color in zip(df['Crop'].unique(), ['#2ecc71','#3498db','#e74c3c','#f39c12']):
    axes[0].hist(df[df['Crop']==crop]['Yield_kg_per_ha'], bins=50, alpha=0.55, label=crop.capitalize(), color=color)
axes[0].set_title('Yield Distribution by Crop', fontweight='bold')
axes[0].set_xlabel('Yield (kg/ha)'); axes[0].legend()

crop_yield = df.groupby('Crop')['Yield_kg_per_ha'].mean().sort_values(ascending=False)
axes[1].bar(crop_yield.index, crop_yield.values, color=['#2ecc71','#3498db','#e74c3c','#f39c12'])
axes[1].set_title('Average Yield by Crop', fontweight='bold')
axes[1].set_ylabel('Avg Yield (kg/ha)')
plt.tight_layout(); plt.show()

In [ ]:
num_cols = ['Area_ha','N_req_kg_per_ha','P_req_kg_per_ha','K_req_kg_per_ha',
            'Temperature_C','Humidity_%','pH','Rainfall_mm','Yield_kg_per_ha']
fig, ax = plt.subplots(figsize=(10, 7))
sns.heatmap(df[num_cols].corr(), annot=True, fmt='.2f', cmap='RdYlGn', center=0, ax=ax)
ax.set_title('Feature Correlation Matrix', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

## 3. Preprocessing

In [ ]:
FEATURE_COLS = ['Year','Area_ha','N_req_kg_per_ha','P_req_kg_per_ha','K_req_kg_per_ha',
                'Temperature_C','Humidity_%','pH','Rainfall_mm',
                'Wind_Speed_m_s','Solar_Radiation_MJ_m2_day','Crop_enc']
TARGET = 'Yield_kg_per_ha'

le = LabelEncoder()
df['Crop_enc'] = le.fit_transform(df['Crop'])
print("Crop encoding:", dict(zip(le.classes_, le.transform(le.classes_))))

X, y = df[FEATURE_COLS], df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)
print(f"Train: {len(X_train):,}  |  Test: {len(X_test):,}")

## 4. Model Training & Evaluation

In [ ]:
models_cfg = {
    'Random Forest':     (RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1), False),
    'Gradient Boosting': (GradientBoostingRegressor(n_estimators=200, learning_rate=0.1, random_state=42), False),
    'Ridge Regression':  (Ridge(alpha=1.0), True),
}

results = {}
for name, (model, scaled) in models_cfg.items():
    Xtr = X_train_sc if scaled else X_train
    Xte = X_test_sc  if scaled else X_test
    model.fit(Xtr, y_train)
    preds = model.predict(Xte)
    results[name] = {
        'model': model, 'preds': preds,
        'RMSE': np.sqrt(mean_squared_error(y_test, preds)),
        'MAE':  mean_absolute_error(y_test, preds),
        'R2':   r2_score(y_test, preds)
    }
    print(f"{name:25s}  RMSE={results[name]['RMSE']:8.2f}  MAE={results[name]['MAE']:8.2f}  R²={results[name]['R2']:.4f}")

## 5. Visualisations

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
fig.suptitle('Actual vs Predicted Yield', fontsize=14, fontweight='bold')
for ax, (name, res), c in zip(axes, results.items(), ['#2ecc71','#3498db','#e74c3c']):
    mn, mx = min(y_test.min(), res['preds'].min()), max(y_test.max(), res['preds'].max())
    ax.scatter(y_test, res['preds'], alpha=0.25, s=7, color=c)
    ax.plot([mn,mx],[mn,mx],'k--',lw=1.5)
    ax.set_title(f"{name}\nR²={res['R2']:.4f}  RMSE={res['RMSE']:.0f}")
    ax.set_xlabel('Actual'); ax.set_ylabel('Predicted')
plt.tight_layout(); plt.show()

In [ ]:
imp = pd.Series(results['Random Forest']['model'].feature_importances_, index=FEATURE_COLS).sort_values()
fig, ax = plt.subplots(figsize=(9,6))
imp.plot(kind='barh', ax=ax, color=['#e74c3c' if v==imp.max() else '#3498db' for v in imp.values])
ax.set_title('Feature Importance — Random Forest', fontweight='bold', fontsize=13)
for i,v in enumerate(imp.values): ax.text(v+0.001, i, f'{v:.4f}', va='center', fontsize=9)
plt.tight_layout(); plt.show()

## 6. Cross-Validation & Save

In [ ]:
best_name = max(results, key=lambda k: results[k]['R2'])
best = results[best_name]
print(f"🏆 Best model: {best_name}  R²={best['R2']:.6f}")

kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv = cross_val_score(best['model'], X, y, cv=kf, scoring='r2', n_jobs=-1)
print(f"CV R² per fold : {[f'{s:.4f}' for s in cv]}")
print(f"Mean R²        : {cv.mean():.4f} ± {cv.std():.4f}")

joblib.dump(best['model'], 'model_output/best_model.pkl')
joblib.dump(scaler,        'model_output/scaler.pkl')
joblib.dump(le,            'model_output/label_encoder.pkl')
print("\n✅ Model saved to model_output/")

## 7. Predict New Samples

In [ ]:
def predict_yield(year, area_ha, N_req, P_req, K_req,
                  temperature, humidity, pH, rainfall,
                  wind_speed, solar_radiation, crop_name):
    """Predict crop yield (kg/ha)."""
    m  = joblib.load('model_output/best_model.pkl')
    le_= joblib.load('model_output/label_encoder.pkl')
    row = np.array([[year, area_ha, N_req, P_req, K_req,
                     temperature, humidity, pH, rainfall,
                     wind_speed, solar_radiation,
                     le_.transform([crop_name.lower()])[0]]])
    return float(m.predict(row)[0])

examples = [
    (2024, 10000, 18.0, 8.0, 11.3, 22, 70, 6.0, 800,  2.5, 20, 'maize',    '🌽'),
    (2024, 50000,  8.4, 4.1,  7.4, 25, 80, 6.5, 1200, 2.0, 18, 'rice',     '🌾'),
    (2024, 20000,  9.0, 5.0,  9.0, 20, 60, 6.5, 600,  1.5, 16, 'chickpea', '🫘'),
    (2024,  5000, 15.0, 7.0, 10.0, 28, 65, 7.0, 500,  3.0, 22, 'cotton',   '🪴'),
]
print(f"{'Crop':10s}  {'Predicted Yield (kg/ha)':>25s}")
print("-"*40)
for *args, crop, icon in examples:
    print(f"{icon} {crop:8s}  {predict_yield(*args, crop_name=crop):>20,.2f}")